In [ ]:
import sys
import importlib
import os

repo_root = os.environ.get("GAP_AWARE_ROOT", os.path.abspath("../../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import model_gap_aware.maskfeat as maskfeat
import model_gap_aware.vivit as vivit_mod
import model_gap_aware.lmdb_training as lmdb_training

importlib.reload(vivit_mod)
importlib.reload(maskfeat)
importlib.reload(lmdb_training)

LMDBConsecutiveDataModule = lmdb_training.LMDBConsecutiveDataModule
build_model = lmdb_training.build_model

from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint, TQDMProgressBar
import pytorch_lightning as pl

In [ ]:
import json

# ---- ESDC LST training config ----
_esdc_data = os.environ.get(
    "ESDC_DATA_DIR",
    "../../../esdc/data/lst_no_syn_gaps",
)
lmdb_path = os.path.join(_esdc_data, "lmdb", "dataset.lmdb")
test_lmdb_path = os.path.join(_esdc_data, "lmdb", "dataset_test.lmdb")

# Derive data shape and normalization range from meta.json
_meta_path = os.path.join(_esdc_data, "meta.json")
if not os.path.exists(_meta_path):
    raise FileNotFoundError(f"meta.json not found at {_meta_path}. Run create_esdc_dataset.py first.")
with open(_meta_path) as _f:
    _meta = json.load(_f)

image_size  = _meta["block_size"][1]
num_frames  = _meta["block_size"][0]
in_channels = 1 + len(_meta.get("aux_vars", []))
norm_range  = tuple(_meta.get("valid_range") or _meta["computed_range"])

vit_patch_size = 8

# Only reconstruct LST (channel 0)
target_channels = [0]
out_channels = len(target_channels)

# Training
batch_size  = 32
num_workers = 4
val_split   = 0.05
max_samples = None  # set to None to use the full dataset (e.g. on HPC)
log_dir     = os.path.join(repo_root, "esdc", "lightning_logs", "pretraining")
max_epochs  = 2
accelerator = "auto"
devices     = "auto"
precision   = "16-mixed"

if not os.path.exists(lmdb_path):
    raise FileNotFoundError(lmdb_path)
if not os.path.exists(test_lmdb_path):
    raise FileNotFoundError(test_lmdb_path)

# ---- LMDB quick check (helps catch key-range issues early) ----
import lmdb

for _path in (lmdb_path, test_lmdb_path):
    _env = lmdb.open(_path, readonly=True, lock=False, readahead=False, meminit=False, subdir=False)
    with _env.begin(write=False) as txn:
        n_entries = int(txn.stat().get("entries", 0))
        if n_entries == 0:
            raise RuntimeError(f"LMDB has 0 entries: {_path}")
        if txn.get(b"0") is None:
            raise RuntimeError(f"LMDB missing key b'0': {_path}")
        if txn.get(str(n_entries - 1).encode()) is None:
            raise RuntimeError(f"LMDB missing last key in: {_path}")
    _env.close()

datamodule = LMDBConsecutiveDataModule(
    lmdb_path=lmdb_path,
    batch_size=batch_size,
    num_workers=num_workers,
    val_split=val_split,
    target_channels=target_channels,
    test_lmdb_path=test_lmdb_path,
    max_samples=max_samples,
)

model = build_model(
    image_size=image_size,
    patch_size=vit_patch_size,
    num_frames=num_frames,
    in_channels=in_channels,
    out_channels=out_channels,
    norm_range=norm_range,
)

callbacks = [
    ModelCheckpoint(
        monitor="MAE_val",
        mode="min",
        save_top_k=1,
        filename="lst-gapfiller-{epoch:02d}-{MAE_val:.4f}",
    ),
    LearningRateMonitor(logging_interval="step"),
    TQDMProgressBar(refresh_rate=50),
]

trainer = pl.Trainer(
    default_root_dir=log_dir,
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=devices,
    precision=precision,
    callbacks=callbacks,
    log_every_n_steps=10,
)

In [ ]:
trainer.fit(model, datamodule=datamodule)

In [ ]:
trainer.test(model, datamodule=datamodule)